In [ ]:
import os

In [16]:
from scipy import io
import os

resnet_101_path = os.path.join(
    "datasets", "AwA2", "xlsa17", "data", "AWA2", "res101.mat"
)

data = io.loadmat(resnet_101_path)

print(data.keys())

print(data["image_files"].shape)

dict_keys(['__header__', '__version__', '__globals__', 'features', 'image_files', 'labels'])
(37322, 1)


# AwA2 incomplete concept sets

Creates a reduced AwA2 predicate matrix (the "incomplete concept set") that a run
consumes via `incomplete=True data.pkl_file_dir=<folder>/`.

The run reads exactly one file from the folder: `predicate-matrix-binary.txt`.
Everything else written alongside it (`info.json`, `concept_names.txt`,
`removed_concepts.npy`, `concept_groups.json`) is for post-hoc analysis.

Nothing here touches the image split — `{train,val,test}_split.npz` is untouched, so a
complete and an incomplete run see exactly the same images and class labels.

In [8]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "datasets" / "awa2_dataset.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from types import SimpleNamespace
import numpy as np

from datasets.awa2_dataset import (
    CONCEPT_SEMANTICS,
    CONCEPT_GROUPS,
    N_CONCEPTS,
    create_custom_incomplete_dataset,
)

# Same rule train.py's update_config_paths applies, so the notebook writes where the
# run will look. On the cluster that is the forced path; locally it is the repo copy.
DEFAULT_DATA_PATH = (
    "/cluster/home/smarcou/work/Data/"
    if "biomed" in os.uname()[1]
    else str(ROOT / "datasets") + "/"
)
print("data_path:", DEFAULT_DATA_PATH)

data_path: /Users/stephenmarcou/Documents/ETH Zurich/Cambridge/Code/SCBM_implementation/datasets/


## The function

`keep` accepts **concept names** (from `CONCEPT_SEMANTICS`) and/or **group names** (from
`CONCEPT_GROUPS`) interchangeably — a group name expands to all of its concepts. Order
and duplicates do not matter; the saved matrix always uses the original AwA2 concept
ordering, which is what the old→new index mapping in `info.json` is keyed on.

In [9]:
def make_incomplete_awa2(keep, data_path=None, incomplete_dir="incomplete_data/"):
    """Create an incomplete AwA2 concept set that keeps `keep`.

    Parameters
    ----------
    keep : iterable of str
        Concept names from CONCEPT_SEMANTICS and/or group names from CONCEPT_GROUPS.
        Group names expand to all concepts in the group.
    data_path : str, optional
        Root holding AwA2/. Defaults to DEFAULT_DATA_PATH.
    incomplete_dir : str
        Subfolder of AwA2/ holding concept sets. Must match config.data.incomplete_dir.

    Returns
    -------
    (folder_name, num_concepts)
        folder_name is what you pass as data.pkl_file_dir=<folder_name>.
    """
    keep = list(keep)
    if not keep:
        raise ValueError("keep is empty; an incomplete set must retain at least one concept.")

    concept_set = set(CONCEPT_SEMANTICS)
    unknown = [k for k in keep if k not in concept_set and k not in CONCEPT_GROUPS]
    if unknown:
        raise ValueError(
            f"Not a concept or group name: {unknown}. "
            f"Concepts: {sorted(concept_set)[:5]}... Groups: {list(CONCEPT_GROUPS)}"
        )

    # Expand group names, then restore original AwA2 ordering.
    keep_idx = set()
    for name in keep:
        if name in CONCEPT_GROUPS:
            keep_idx.update(CONCEPT_GROUPS[name])
        else:
            keep_idx.add(CONCEPT_SEMANTICS.index(name))
    keep_names = [CONCEPT_SEMANTICS[i] for i in sorted(keep_idx)]

    if len(keep_names) == N_CONCEPTS:
        raise ValueError("keep covers all 85 concepts; that is the complete set, not an incomplete one.")

    hidden = [c for c in CONCEPT_SEMANTICS if c not in set(keep_names)]
    print(f"keeping {len(keep_names)} concepts, hiding {len(hidden)}: {hidden}\n")

    cfg = SimpleNamespace(
        data_path=data_path or DEFAULT_DATA_PATH,
        incomplete_dir=incomplete_dir,
    )
    folder, n = create_custom_incomplete_dataset(cfg, keep_names)

    print(f"\n  data.pkl_file_dir={folder}")
    print(f"  rsync -av {os.path.join(cfg.data_path, 'AwA2', incomplete_dir, folder.rstrip('/'))} \\")
    print(f"    smarcou@<cluster>:/cluster/home/smarcou/work/Data/AwA2/incomplete_data/")
    return folder, n

In [ ]:
keep = ["black", "gray", "stripes", "hairless", "flippers", "paws", "plains", "fierce", "solitary"]

#make_incomplete_awa2(keep, data_path=DEFAULT_DATA_PATH, incomplete_dir="incomplete_data/")


keeping 9 concepts, hiding 76: ['white', 'blue', 'brown', 'orange', 'red', 'yellow', 'patches', 'spots', 'furry', 'toughskin', 'big', 'small', 'bulbous', 'lean', 'hands', 'hooves', 'pads', 'longleg', 'longneck', 'tail', 'chewteeth', 'meatteeth', 'buckteeth', 'strainteeth', 'horns', 'claws', 'tusks', 'smelly', 'flys', 'hops', 'swims', 'tunnels', 'walks', 'fast', 'slow', 'strong', 'weak', 'muscle', 'bipedal', 'quadrapedal', 'active', 'inactive', 'nocturnal', 'hibernate', 'agility', 'fish', 'meat', 'plankton', 'vegetation', 'insects', 'forager', 'grazer', 'hunter', 'scavenger', 'skimmer', 'stalker', 'newworld', 'oldworld', 'arctic', 'coastal', 'desert', 'bush', 'forest', 'fields', 'jungle', 'mountains', 'ocean', 'ground', 'water', 'tree', 'cave', 'timid', 'smart', 'group', 'nestspot', 'domestic']

Keeping AwA2 concepts:
  0: black
  4: gray
  10: stripes
  12: hairless
  18: flippers
  22: paws
  68: plains
  78: fierce
  82: solitary
Saved incomplete AwA2 concept set to /Users/stephenmar

('awa2_incomplete_local_custom_1/', 9)

## Reference: the 28 semantic groups

In [11]:
for g, idx in CONCEPT_GROUPS.items():
    print(f"{g:22s} {len(idx):3d}  {', '.join(CONCEPT_SEMANTICS[i] for i in idx)}")

color                    8  black, white, blue, brown, gray, orange, red, yellow
fur_pattern              6  patches, spots, stripes, furry, hairless, toughskin
size                     4  big, small, bulbous, lean
limb_shape               7  flippers, hands, hooves, pads, paws, longleg, longneck
tail                     1  tail
teeth_type               4  chewteeth, meatteeth, buckteeth, strainteeth
horns                    1  horns
claws                    1  claws
tusks                    1  tusks
smelly                   1  smelly
transport_mechanism      5  flys, hops, swims, tunnels, walks
speed                    2  fast, slow
strength                 2  strong, weak
muscle                   1  muscle
movement_move            2  bipedal, quadrapedal
active                   2  active, inactive
nocturnal                1  nocturnal
hibernate                1  hibernate
agility                  1  agility
diet                     5  fish, meat, plankton, vegetation, insects
feedin

## Usage

Keep an explicit list of concepts:

```python
folder, n = make_incomplete_awa2(["black", "white", "big", "small", "tail", "claws"])
```

Keep whole groups by name:

```python
folder, n = make_incomplete_awa2(["color", "size", "fur_pattern", "limb_shape"])
```

Hide a group (keep everything else) — the complement idiom:

```python
HIDE = {"biome"}
folder, n = make_incomplete_awa2([g for g in CONCEPT_GROUPS if g not in HIDE])
```

In [12]:
# folder, n = make_incomplete_awa2([g for g in CONCEPT_GROUPS if g not in {"biome"}])

## Verify

The run derives `data.num_concepts` from this matrix's column count, so this is the
whole run-side contract: 50 rows, `n` columns.

In [13]:
def check_incomplete_awa2(folder, data_path=None, incomplete_dir="incomplete_data/"):
    root = os.path.join(data_path or DEFAULT_DATA_PATH, "AwA2", incomplete_dir, folder.rstrip("/"))
    M = np.genfromtxt(os.path.join(root, "predicate-matrix-binary.txt"), dtype=int)
    removed = np.load(os.path.join(root, "removed_concepts.npy"))
    print(f"{root}\n  matrix {M.shape}  ->  data.num_concepts={M.shape[1]}")
    print(f"  hidden ({len(removed)}): {[CONCEPT_SEMANTICS[i] for i in removed]}")
    return M

# check_incomplete_awa2(folder)